In [1]:
!python -V

Python 3.12.10


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error,root_mean_squared_error

In [6]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/heodi/Projects/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1747689092171, experiment_id='2', last_update_time=1747689092171, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [7]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [8]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [9]:
len(df_train), len(df_val)

(73908, 61921)

In [10]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [11]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [12]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: divide by zero encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: overflow encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: invalid value encountered in matmul
  intercept_ = y_offset - X_offset @ coef_


60.197661629133236

In [21]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [24]:
with mlflow.start_run():

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: divide by zero encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: overflow encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: invalid value encountered in matmul
  intercept_ = y_offset - X_offset @ coef_


In [14]:
import xgboost as xgb

In [15]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [16]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [31]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [32]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:6.85120                           
[1]	validation-rmse:6.71059                           
[2]	validation-rmse:6.69338                           
[3]	validation-rmse:6.67866                           
[4]	validation-rmse:6.66844                           
  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [22:45:22] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:250: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[5]	validation-rmse:6.65707                           
[6]	validation-rmse:6.65123                           
[7]	validation-rmse:6.64562                           
[8]	validation-rmse:6.63875                           
[9]	validation-rmse:6.63282                           
[10]	validation-rmse:6.62880                          
[11]	validation-rmse:6.62488                          
[12]	validation-rmse:6.62167                          
[13]	validation-rmse:6.61776                          
[14]	validation-rmse:6.61316                          
[15]	validation-rmse:6.60975                          
[16]	validation-rmse:6.60293                          
[17]	validation-rmse:6.60128                          
[18]	validation-rmse:6.59992                          
[19]	validation-rmse:6.59849                          
[20]	validation-rmse:6.59753                          
[21]	validation-rmse:6.59545                          
[22]	validation-rmse:6.59514                          
[23]	valid

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [22:45:36] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:250: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[4]	validation-rmse:10.30603                                                   
[5]	validation-rmse:10.00530                                                   
[6]	validation-rmse:9.72777                                                    
[7]	validation-rmse:9.47220                                                    
[8]	validation-rmse:9.23681                                                    
[9]	validation-rmse:9.02044                                                    
[10]	validation-rmse:8.82162                                                   
[11]	validation-rmse:8.63922                                                   
[12]	validation-rmse:8.47206                                                   
[13]	validation-rmse:8.31894                                                   
[14]	validation-rmse:8.17893                                                   
[15]	validation-rmse:8.05080                                                   
[16]	validation-rmse:7.93401            

...

[8]	validation-rmse:6.70947                                                     
[9]	validation-rmse:6.69888                                                     
[10]	validation-rmse:6.68934                                                    
[11]	validation-rmse:6.68233                                                    
[12]	validation-rmse:6.68068                                                    
[13]	validation-rmse:6.67603                                                    
[14]	validation-rmse:6.66972                                                    
[15]	validation-rmse:6.66691                                                    
[16]	validation-rmse:6.66346                                                    
[17]	validation-rmse:6.66042                                                    
[18]	validation-rmse:6.65778                                                    
[19]	validation-rmse:6.65435                                                    
[20]	validation-rmse:6.65265

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [23:31:24] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:250: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:8.14098                                                     
[1]	validation-rmse:6.97066                                                     
[2]	validation-rmse:6.65014                                                     
[3]	validation-rmse:6.54759                                                     
[4]	validation-rmse:6.50250                                                     
[5]	validation-rmse:6.48132                                                     
[6]	validation-rmse:6.47394                                                     
[7]	validation-rmse:6.47020                                                     
[8]	validation-rmse:6.46447                                                     
[9]	validation-rmse:6.45716                                                     
[10]	validation-rmse:6.45409                                                    
[11]	validation-rmse:6.44924                                                    
[12]	validation-rmse:6.44566

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [23:31:31] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:250: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:10.05306                                                    
[1]	validation-rmse:8.69407                                                     
[2]	validation-rmse:7.85045                                                     
[3]	validation-rmse:7.34806                                                     
[4]	validation-rmse:7.04575                                                     
[5]	validation-rmse:6.86259                                                     
[6]	validation-rmse:6.75298                                                     
[7]	validation-rmse:6.67722                                                     
[8]	validation-rmse:6.62990                                                     
[9]	validation-rmse:6.59746                                                     
[10]	validation-rmse:6.56840                                                    
[11]	validation-rmse:6.55391                                                    
[12]	validation-rmse:6.54341

In [18]:
mlflow.xgboost.autolog(disable=True, model_foramt='json')

In [19]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [21:28:38] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:250: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44482
[1]	validation-rmse:10.77202
[2]	validation-rmse:10.18363
[3]	validation-rmse:9.67396
[4]	validation-rmse:9.23166
[5]	validation-rmse:8.84808
[6]	validation-rmse:8.51883
[7]	validation-rmse:8.23597
[8]	validation-rmse:7.99320
[9]	validation-rmse:7.78709
[10]	validation-rmse:7.61022
[11]	validation-rmse:7.45952
[12]	validation-rmse:7.33049
[13]	validation-rmse:7.22098
[14]	validation-rmse:7.12713
[15]	validation-rmse:7.04752
[16]	validation-rmse:6.98005
[17]	validation-rmse:6.92232
[18]	validation-rmse:6.87112
[19]	validation-rmse:6.82740
[20]	validation-rmse:6.78995
[21]	validation-rmse:6.75792
[22]	validation-rmse:6.72994
[23]	validation-rmse:6.70547
[24]	validation-rmse:6.68390
[25]	validation-rmse:6.66421
[26]	validation-rmse:6.64806
[27]	validation-rmse:6.63280
[28]	validation-rmse:6.61924
[29]	validation-rmse:6.60773
[30]	validation-rmse:6.59777
[31]	validation-rmse:6.58875
[32]	validation-rmse:6.58107
[33]	validation-rmse:6.57217
[34]	validation-rmse:

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/mlflow/xgboost/__init__.py:168: UserWarning: [21:29:08] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)
2025/05/22 21:29:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [39]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        

/Users/heodi/Projects/mlops-zoomcamp/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [20]:
logged_model = 'runs:/6ac3dfc62d1243b1b78ba18aa03060d7/models_mlflow'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)


In [21]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: models_mlflow
  flavor: mlflow.xgboost
  run_id: 6ac3dfc62d1243b1b78ba18aa03060d7

In [22]:
xgboost_model = mlflow.xgboost.load_model(logged_model)

In [25]:
y_pred=xgboost_model.predict(valid)

In [29]:
y_pred[:10]

array([14.782765 ,  7.184751 , 15.971323 , 24.328938 ,  9.559302 ,
       17.115105 , 11.6522455,  8.688133 ,  8.962229 , 18.982166 ],
      dtype=float32)

In [30]:
y_val[:10]

array([17.91666667,  6.5       , 15.25      , 18.23333333,  8.96666667,
        7.85      ,  9.7       , 11.28333333,  8.73333333,  1.71666667])